In [0]:
# 1. Setup
from pyspark.sql import functions as F

SOURCE_PATH = "/databricks-datasets/retail-org/suppliers/"
TARGET_TABLE = "retail_dev.bronze.bronze_suppliers"


In [0]:
# 2. Schema e Volume
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ",")
    .csv(SOURCE_PATH)
)

print("=== SCHEMA ===")
df.printSchema()

print(f"\n=== VOLUME ===")
print(f"Total de linhas:   {df.count()}")
print(f"Total de colunas:  {len(df.columns)}")
print(f"Colunas:           {df.columns}")


In [0]:
# 3. Amostra
display(df.limit(10))


In [0]:
# 4. Qualidade — nulos por coluna
display(
    df.select([
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ])
)

In [0]:
# 4b. Fornecedores duplicados?
id_col = next((c for c in df.columns if "id" in c.lower()), df.columns[0])

display(
    df.groupBy(id_col)
    .agg(F.count("*").alias("ocorrencias"))
    .filter(F.col("ocorrencias") > 1)
    .orderBy(F.col("ocorrencias").desc())
)


In [0]:
# 5. Análise — distribuição geográfica
geo_cols = [c for c in df.columns if any(k in c.lower() for k in ["state", "city", "country", "region", "estado", "cidade"])]

if geo_cols:
    for col in geo_cols:
        print(f"\n=== {col} ===")
        display(
            df.groupBy(col)
            .agg(F.count("*").alias("total"))
            .orderBy(F.col("total").desc())
        )
else:
    print("Nenhuma coluna geográfica identificada. Colunas disponíveis:")
    for c in df.columns:
        print(f"  {c}")

# COMMAND ----------

# 5b. Distribuição por colunas categóricas (2 a 50 valores distintos)
for c in df.columns:
    distinct_count = df.select(c).distinct().count()
    if 2 <= distinct_count <= 50:
        print(f"\n=== {c} ({distinct_count} valores distintos) ===")
        display(
            df.groupBy(c)
            .agg(F.count("*").alias("total"))
            .orderBy(F.col("total").desc())
        )

In [0]:
# 6. Ingestão → Delta
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Tabela escrita: {TARGET_TABLE}")

# COMMAND ----------

# 7. Verificação
df_check = spark.table(TARGET_TABLE)

print(f"Linhas na tabela: {df_check.count()}")
print(f"Colunas:          {df_check.columns}")

# COMMAND ----------

display(df_check.limit(5))
